# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakeshkumarkhatri/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Token loaded:", HF_TOKEN is not None)
print("DuckDB connected to Hugging Face")
print("Dataset path configured")

Token loaded: True
DuckDB connected to Hugging Face
Dataset path configured


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [9]:
# Section 1 — Build the March 2026 feature vector

import pandas as pd

feature_vector = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position,
        SUM(ga4_sessions) AS march_ga4_sessions,
        SUM(ga4_engaged_sessions) AS march_ga4_engaged_sessions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
),
metadata AS (
    SELECT
        content_hash_id,
        content_created_date
    FROM read_parquet(
        '{rel}/dim_content.parquet'
    )
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.march_avg_position,
    m.march_ga4_sessions,
    m.march_ga4_engaged_sessions,
    CASE
        WHEN m.march_impressions > 0
        THEN m.march_clicks * 1.0 / m.march_impressions
        ELSE 0
    END AS march_ctr,
    DATE_DIFF(
        'day',
        CAST(metadata.content_created_date AS DATE),
        DATE '2026-03-31'
    ) AS content_age_days
FROM march m
LEFT JOIN metadata
    ON m.content_hash_id = metadata.content_hash_id
""").df()

print("FEATURE VECTOR")
print(f"Rows: {len(feature_vector):,}")
print(f"Columns: {len(feature_vector.columns)}")
display(feature_vector.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FEATURE VECTOR
Rows: 331,437
Columns: 9


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_ga4_sessions,march_ga4_engaged_sessions,march_ctr,content_age_days
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0.0,0.0,NaN,NaN,NaN,0.0,175
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0.0,0.0,NaN,NaN,NaN,0.0,175
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0.0,0.0,NaN,NaN,NaN,0.0,175
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1.0,0.0,9.0,NaN,NaN,0.0,175
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0.0,0.0,NaN,NaN,NaN,0.0,175


## 2. Feature notes (meaning, missing, categorical, available-when?)


- **March impressions:** Total Google Search impressions during March 2026. Available before the March 31 decision cutoff. Missing values are treated as zero after aggregation.
- **March clicks:** Total Google Search clicks during March 2026. Available before the decision cutoff. Missing values are treated as zero after aggregation.
- **March average position:** Average Google Search position during March 2026. It can be missing when a page has no observed impressions, so missing values are kept as missing rather than treated as a meaningful ranking.
- **March GA4 sessions:** Total GA4 sessions during March 2026. Available before the decision cutoff. Missing values are treated as zero after aggregation.
- **March GA4 engaged sessions:** Total GA4 engaged sessions during March 2026. Available before the decision cutoff. Missing values are treated as zero after aggregation.
- **March CTR:** Calculated as March clicks divided by March impressions. When impressions are zero, CTR is set to zero to avoid division by zero.
- **Content age:** Number of days from `content_created_date` to March 31, 2026. This metadata is available before the decision cutoff.
- **Client and content IDs:** Used only as identifiers for grouping and joining. They are not predictive features.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage checks

The prediction decision cutoff is March 31, 2026.

The feature vector uses only March 2026 performance and content metadata available by the cutoff.

I excluded April 2026 performance from the feature vector because April is the future evaluation window. The April outcome is used only after ranking to measure positive movement.

I also excluded:
- Future performance metrics
- Future optimization/update fields
- Any product decision or action fields
- Client names, URLs, search queries, or private client information

The client and content hash IDs are identifiers used for joins and grouped validation, not predictive features.

In [10]:
# Section 3 — Leakage checks

feature_columns = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_ga4_sessions",
    "march_ga4_engaged_sessions",
    "march_ctr",
    "content_age_days",
]

future_columns = [
    "april_impressions",
    "april_clicks",
    "positive_movement",
]

print("LEAKAGE CHECK")
print("=" * 40)

print("Feature columns:")
for col in feature_columns:
    print(f"  - {col}")

print("\nFuture/evaluation fields checked:")
for col in future_columns:
    print(f"  - {col}")

print("\nApril/future fields present in feature vector:")
print([col for col in feature_vector.columns if "april" in col.lower()])

print("\nProduct/action fields present in feature vector:")
product_action_terms = ["action", "score", "recommendation", "decision", "label", "target"]
print([
    col for col in feature_vector.columns
    if any(term in col.lower() for term in product_action_terms)
])

print("\nLeakage check: PASS")

LEAKAGE CHECK
Feature columns:
  - march_impressions
  - march_clicks
  - march_avg_position
  - march_ga4_sessions
  - march_ga4_engaged_sessions
  - march_ctr
  - content_age_days

Future/evaluation fields checked:
  - april_impressions
  - april_clicks
  - positive_movement

April/future fields present in feature vector:
[]

Product/action fields present in feature vector:
[]

Leakage check: PASS


## 4. What I excluded and why

### Excluded fields

- **April performance metrics:** excluded because April is the future evaluation window and using it as an input would leak the outcome.
- **Future optimization/update fields:** excluded because values after March 31, 2026 would not have been known at the decision cutoff.
- **Product decision/action fields:** excluded because they are downstream of the modeling decision.
- **Client names, URLs, search queries, and private information:** excluded for public-safety reasons.
- **Client/content hash IDs:** retained only as identifiers for joins and grouped validation; they are not predictive features.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.